# Makine Öğrenmesi: Tadım notundan puan tahmini

## 1-verinin BigQuery'den çekilmesi

In [2]:
from google.colab import auth
auth.authenticate_user()

In [3]:
from google.cloud import bigquery
import pandas as pd

PROJECT_ID = "wine-analytics-508220"
client = bigquery.Client(project=PROJECT_ID)

sorgu = """
select review_id, description, points, price
from `wine-analytics-508220.dbt_wine.fct_wine_review`
"""

df = client.query(sorgu, location="europe-west1").to_dataframe()

print("Satır:", len(df))
df.head()

Satır: 129907


,review_id,description,points,price
0,2386,"Way too much banana, mango and sugar. Not a pr...",80,10.0
1,2389,"Heavy hayfield aromas are oxidized, and the fl...",80,12.0
2,2699,"Medicinal and feral on the nose, then heavy, t...",80,12.0
3,2963,Strange Chardonnay. It smells and tastes more ...,80,9.0
4,2989,Just when you think Alta Vista has gotten ever...,96,48.0


##2-Verinin eğitim test olarak ikiye bölünmesi

Verinin %80'iyle eğitiyoruz, %20'sini hiç göstermeden saklıyoruz. Sonunda o saklı kısımla test ediyoruz
X girdi, yani tadım notları. y tahmin etmek istediğimiz şey, yani puan.

In [4]:
from sklearn.model_selection import train_test_split

X = df["description"]
y = df["points"]

X_egitim, X_test, y_egitim, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Eğitim:", len(X_egitim))
print("Test:", len(X_test))

Eğitim: 103925
Test: 25982


##3-Kıyas noktası(baseline) oluşturma


In [5]:
from sklearn.metrics import mean_absolute_error
import numpy as np

tahmin_basit = np.full(len(y_test), y_egitim.mean())
hata_basit = mean_absolute_error(y_test, tahmin_basit)

print("Basit tahmin hatası:", round(hata_basit, 2), "puan")

Basit tahmin hatası: 2.48 puan


4- Metni sayıya çevirmek

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

vektorlestirici = TfidfVectorizer(
    max_features=5000,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=5
)

X_egitim_vek = vektorlestirici.fit_transform(X_egitim)
X_test_vek = vektorlestirici.transform(X_test)

print("Eğitim:", X_egitim_vek.shape)
print("Test:", X_test_vek.shape)

Eğitim: (103925, 5000)
Test: (25982, 5000)


5-Modeli eğit

In [7]:
from sklearn.linear_model import Ridge

model = Ridge(alpha=1.0)
model.fit(X_egitim_vek, y_egitim)

tahmin_model = model.predict(X_test_vek)
hata_model = mean_absolute_error(y_test, tahmin_model)

print("Basit tahmin hatası:", round(hata_basit, 2), "puan")
print("Model hatası:       ", round(hata_model, 2), "puan")

Basit tahmin hatası: 2.48 puan
Model hatası:        1.34 puan


6-Model ne öğrendi?

In [8]:
kelimeler = vektorlestirici.get_feature_names_out()
agirliklar = model.coef_

tablo = pd.DataFrame({"kelime": kelimeler, "agirlik": agirliklar})

print("PUANI EN ÇOK YÜKSELTEN KELİMELER")
print(tablo.sort_values("agirlik", ascending=False).head(15).to_string(index=False))

print()
print("PUANI EN ÇOK DÜŞÜREN KELİMELER")
print(tablo.sort_values("agirlik").head(15).to_string(index=False))

PUANI EN ÇOK YÜKSELTEN KELİMELER
   kelime  agirlik
   superb 7.825907
     2030 7.760293
 gorgeous 7.648897
       93 7.401424
beautiful 7.244728
    years 7.160329
 stunning 7.125242
       94 7.114749
     2026 7.035731
  decades 6.829769
  amazing 6.690068
     2025 6.690063
     2022 6.529240
 dazzling 6.492703
delicious 6.491192

PUANI EN ÇOK DÜŞÜREN KELİMELER
          kelime   agirlik
           lacks -4.981687
          watery -4.597736
         strange -4.145287
           mealy -3.990587
             odd -3.665956
golden delicious -3.422698
            sour -3.421311
          simple -3.180756
         lacking -3.104560
            lack -3.095956
       puckering -3.009644
         vegetal -2.974333
           weedy -2.939208
         curious -2.876297
            dull -2.871537


### Hipotez Testleri

In [9]:
sorgu_test = """
select c.country, c.province, f.points, f.price
from `wine-analytics-508220.dbt_wine.fct_wine_review` f
join `wine-analytics-508220.dbt_wine.dim_country` c
  on f.country_id = c.country_id
"""

dft = client.query(sorgu_test, location="europe-west1").to_dataframe()
print(len(dft))

129907


Test 1: Avusturya vs Arjantin

In [10]:
from scipy import stats

avusturya = dft[dft["country"] == "Austria"]["points"]
arjantin = dft[dft["country"] == "Argentina"]["points"]

t, p = stats.ttest_ind(avusturya, arjantin, equal_var=False)

print(f"Avusturya ortalama: {avusturya.mean():.2f}  (n={len(avusturya)})")
print(f"Arjantin ortalama:  {arjantin.mean():.2f}  (n={len(arjantin)})")
print(f"p-değeri: {p:.2e}")

Avusturya ortalama: 90.10  (n=3345)
Arjantin ortalama:  86.71  (n=3800)
p-değeri: 0.00e+00


Test 2: Alsace vs Burgundy

In [11]:
alsace = dft[dft["province"] == "Alsace"]["points"]
burgundy = dft[dft["province"] == "Burgundy"]["points"]

t2, p2 = stats.ttest_ind(alsace, burgundy, equal_var=False)

print(f"Alsace:   {alsace.mean():.2f}  (n={len(alsace)})")
print(f"Burgundy: {burgundy.mean():.2f}  (n={len(burgundy)})")
print(f"p-değeri: {p2:.4f}")

Alsace:   89.37  (n=2440)
Burgundy: 89.57  (n=3980)
p-değeri: 0.0111


Veri sızıntısı kontrolü. Model 93 ve 94 kelimelerine yüksek ağırlık verdi. Notların içinde puan yazıyorsa model kopya çekiyor olabilir.Kontrol: notlarda açık puan ifadesi ("93 points" gibi) kaç kez geçtiği kontrol edildi. 9 çıktığı için öyle bir durum olmadığı görüldü.

In [13]:
import re

kalip = re.compile(r"\b(8[0-9]|9[0-9]|100)\s*(points?|pts|/\s*100)\b", re.I)
sayi = df["description"].str.contains(kalip, na=False).sum()

print(f"Notunda açıkça puan geçen kayıt: {sayi} / {len(df)}")

/tmp/ipykernel_870/3384698589.py:4: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  sayi = df["description"].str.contains(kalip, na=False).sum()


Notunda açıkça puan geçen kayıt: 9 / 129907
